In [ ]:
!pip install doubleml xgboost linearmodels matplotlib seaborn pandas numpy scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import doubleml as dml
from doubleml.datasets import fetch_401K
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from linearmodels.iv import IV2SLS

# 1. Carga de Datos
data_401k = fetch_401K(return_type='DataFrame')
obj_y, obj_d, obj_z = 'net_tfa', 'p401', 'e401'
obj_x = ['inc', 'age', 'educ', 'fsize', 'marr', 'twoearn', 'pira', 'hown']

# 2. Generar Tabla Descriptiva para el Anexo
desc_stats = data_401k[[obj_y, obj_d, obj_z] + obj_x].describe().T
desc_stats.to_csv('tabla_descriptiva_anexo.csv') # Guarda un CSV para LaTeX
print("✅ Datos cargados. Tabla descriptiva guardada como 'tabla_descriptiva_anexo.csv'.")
desc_stats[['mean', 'std', 'min', '50%', 'max']]

In [ ]:
print("📊 ESTIMACIÓN CLÁSICA Y TEST DE SUPUESTOS")

data_401k['const'] = 1
formula_iv = f"{obj_y} ~ 1 + {' + '.join(obj_x)} + [{obj_d} ~ {obj_z}]"

# Corremos IV clásico
modelo_iv = IV2SLS.from_formula(formula_iv, data_401k).fit(cov_type='robust')

# Extraemos el test de la primera etapa (Relevancia del instrumento)
# CORRECCIÓN AQUÍ: Se usa 'f.stat' con punto
f_stat = modelo_iv.first_stage.diagnostics.loc['p401', 'f.stat']
print(f"🔥 Estadístico F (Primera Etapa): {f_stat:.2f}")
if f_stat > 10:
    print("   -> ¡Excelente! El instrumento es fuerte (F > 10). Supuesto validado.")

coef_iv = modelo_iv.params[obj_d]
ci_iv = modelo_iv.conf_int().loc[obj_d].values
print(f"⚠️ Efecto 2SLS Clásico: {coef_iv:.2f} | IC: [{ci_iv[0]:.2f}, {ci_iv[1]:.2f}]")

In [ ]:
print("⏳ Entrenando Modelos de Machine Learning (Puede tomar 1-2 minutos)...")

dml_data_iv = dml.DoubleMLData(data_401k, y_col=obj_y, d_cols=obj_d, z_cols=obj_z, x_cols=obj_x)

ml_l = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
ml_m = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
ml_r = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

# Modelo Principal (RF)
dml_pliv_rf = dml.DoubleMLPLIV(dml_data_iv, ml_l=ml_l, ml_m=ml_m, ml_r=ml_r, n_folds=3).fit()
coef_rf = dml_pliv_rf.coef[0]
ci_rf = dml_pliv_rf.confint().iloc[0].values

# Robustez (XGB)
ml_l_xgb = XGBRegressor(n_estimators=100, max_depth=3, random_state=42, learning_rate=0.1)
ml_m_xgb = XGBRegressor(n_estimators=100, max_depth=3, random_state=42, learning_rate=0.1)
ml_r_xgb = XGBRegressor(n_estimators=100, max_depth=3, random_state=42, learning_rate=0.1)
dml_pliv_xgb = dml.DoubleMLPLIV(dml_data_iv, ml_l=ml_l_xgb, ml_m=ml_m_xgb, ml_r=ml_r_xgb, n_folds=3).fit()
coef_xgb = dml_pliv_xgb.coef[0]
ci_xgb = dml_pliv_xgb.confint().iloc[0].values

print(f"✅ DML (Random Forest): {coef_rf:.2f} | IC: [{ci_rf[0]:.2f}, {ci_rf[1]:.2f}]")
print(f"✅ DML (XGBoost):       {coef_xgb:.2f} | IC: [{ci_xgb[0]:.2f}, {ci_xgb[1]:.2f}]")

In [ ]:
print("⏳ Escaneando Desigualdad Demográfica...")

def estimar_subgrupo(df, nombre):
    dml_sub = dml.DoubleMLData(df, y_col=obj_y, d_cols=obj_d, z_cols=obj_z, x_cols=obj_x)
    modelo = dml.DoubleMLPLIV(dml_sub, ml_l=ml_l, ml_m=ml_m, ml_r=ml_r, n_folds=3).fit()
    return modelo.coef[0], modelo.confint().iloc[0].values

# Por Ingreso
med_inc = data_401k['inc'].median()
c_bajos, ci_bajos = estimar_subgrupo(data_401k[data_401k['inc'] <= med_inc], "Bajos Ingresos")
c_altos, ci_altos = estimar_subgrupo(data_401k[data_401k['inc'] > med_inc], "Altos Ingresos")

# Por Edad
med_age = data_401k['age'].median()
c_jov, ci_jov = estimar_subgrupo(data_401k[data_401k['age'] <= med_age], "Jóvenes")
c_adu, ci_adu = estimar_subgrupo(data_401k[data_401k['age'] > med_age], "Adultos")

# Guardar resultados en DataFrame para exportar
res_het = pd.DataFrame({
    'Grupo': ['Bajos Ingresos', 'Altos Ingresos', 'Jóvenes', 'Adultos Mayores'],
    'Coeficiente': [c_bajos, c_altos, c_jov, c_adu],
    'IC_Inf': [ci_bajos[0], ci_altos[0], ci_jov[0], ci_adu[0]],
    'IC_Sup': [ci_bajos[1], ci_altos[1], ci_jov[1], ci_adu[1]]
})
res_het.to_csv('tabla_heterogenea.csv', index=False)
print("✅ Efectos condicionales calculados y guardados en 'tabla_heterogenea.csv'.")

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# GRÁFICO 1: Comparación Metodológica (Para la sección principal)
modelos_main = ['2SLS (Clásico)', 'DML (RF)', 'DML (XGB)']
coefs_main = [coef_iv, coef_rf, coef_xgb]
err_main = [[coef_iv-ci_iv[0], coef_rf-ci_rf[0], coef_xgb-ci_xgb[0]],
            [ci_iv[1]-coef_iv, ci_rf[1]-coef_rf, ci_xgb[1]-coef_xgb]]

axes[0].errorbar(modelos_main, coefs_main, yerr=err_main, fmt='o', color='#2c3e50',
                 ecolor='#e74c3c', elinewidth=3, capsize=6, markersize=8)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Fig 1: Efecto LATE Global (Clásico vs DML)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Impacto en Riqueza ($)', fontsize=12)

# GRÁFICO 2: Efectos Heterogéneos (Para la sección de desigualdad)
grupos_het = res_het['Grupo'].tolist()
coefs_het = res_het['Coeficiente'].tolist()
err_het = [[c - inf for c, inf in zip(coefs_het, res_het['IC_Inf'])],
           [sup - c for c, sup in zip(coefs_het, res_het['IC_Sup'])]]

axes[1].errorbar(grupos_het, coefs_het, yerr=err_het, fmt='s', color='#2980b9',
                 ecolor='#27ae60', elinewidth=3, capsize=6, markersize=8)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Fig 2: Efectos Heterogéneos (DML-RF)', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('graficos_paper_401k.png', dpi=300)
print("📸 ¡Gráficos maestros generados y guardados como 'graficos_paper_401k.png'!")
plt.show()

In [ ]:
!pip install econml graphviz

In [ ]:
from econml.iv.dml import OrthoIV
from econml.cate_interpreter import SingleTreeCateInterpreter
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

print("⏳ Entrenando el motor Causal de EconML (Buscando interacciones ocultas)...")

# 1. Separar las variables en el formato que exige EconML
Y = data_401k[obj_y]  # Resultado
T = data_401k[obj_d]  # Tratamiento
Z = data_401k[obj_z]  # Instrumento
X = data_401k[obj_x]  # Controles (Donde el algoritmo buscará los cortes)

# 2. Configurar el modelo IV-DML de EconML
modelo_econml = OrthoIV(
    model_y_xw=RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    model_t_xw=RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    model_z_xw=RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    discrete_treatment=True,
    discrete_instrument=True,
    random_state=42
)

# Ajustar el modelo a los datos
modelo_econml.fit(Y, T, Z=Z, X=X)

print("🌳 Construyendo el Árbol Causal de descubrimientos...")

# 3. Llamar al Intérprete para que nos traduzca la caja negra a un árbol simple
# max_depth=2 significa que hará máximo 2 niveles de cortes para que sea fácil de leer
interprete = SingleTreeCateInterpreter(include_model_uncertainty=True, max_depth=2)
interprete.interpret(modelo_econml, X)

# 4. Dibujar y guardar el árbol
plt.figure(figsize=(16, 8))
interprete.plot(feature_names=obj_x, fontsize=12)
plt.title("Árbol Causal: Descubrimiento Automático de Desigualdad en el 401(k)", fontsize=16, fontweight='bold')
plt.tight_layout()

# Guardamos la imagen
plt.savefig('arbol_causal_401k.png', dpi=300, bbox_inches='tight')
print("📸 ¡Árbol Causal generado exitosamente y guardado como 'arbol_causal_401k.png'!")
plt.show()

In [ ]:
# =============================================================================
# SALIDAS ESTILO STATA PARA EL ANEXO DEL PAPER
# =============================================================================

print("\n" + "="*80)
print(" 🏛️ TABLA 1: ESTIMACIÓN CLÁSICA (2SLS / Variables Instrumentales)")
print("="*80)
# La librería linearmodels tiene un .summary idéntico a STATA
print(modelo_iv.summary)


print("\n\n" + "="*80)
print(" 🤖 TABLA 2: ESTIMACIÓN DOUBLE MACHINE LEARNING (Random Forest)")
print("="*80)
# En DoubleML, .summary es un DataFrame (van SIN paréntesis)
print(dml_pliv_rf.summary)


print("\n\n" + "="*80)
print(" 🤖 TABLA 3: ESTIMACIÓN DOUBLE MACHINE LEARNING (XGBoost Robustez)")
print("="*80)
# Sin paréntesis
print(dml_pliv_xgb.summary)

In [ ]:
from econml.iv.dml import OrthoIV
from econml.cate_interpreter import SingleTreeCateInterpreter
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import os
import zipfile
from google.colab import files

print("⏳ Entrenando el motor Causal de EconML (Buscando interacciones ocultas)...")

# 1. Separar las variables en el formato que exige EconML
Y = data_401k[obj_y]  # Resultado
T = data_401k[obj_d]  # Tratamiento
Z = data_401k[obj_z]  # Instrumento
X = data_401k[obj_x]  # Controles (Donde el algoritmo buscará los cortes)

# 2. Configurar el modelo IV-DML de EconML
modelo_econml = OrthoIV(
    model_y_xw=RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    model_t_xw=RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    model_z_xw=RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    discrete_treatment=True,
    discrete_instrument=True,
    random_state=42
)

# Ajustar el modelo a los datos
modelo_econml.fit(Y, T, Z=Z, X=X)

print("🌳 Construyendo el Árbol Causal de descubrimientos...")

# 3. Llamar al Intérprete para que nos traduzca la caja negra a un árbol simple
# max_depth=2 significa que hará máximo 2 niveles de cortes para que sea fácil de leer
interprete = SingleTreeCateInterpreter(include_model_uncertainty=True, max_depth=2)
interprete.interpret(modelo_econml, X)

# 4. Dibujar y guardar el árbol
plt.figure(figsize=(16, 8))
interprete.plot(feature_names=obj_x, fontsize=12)
plt.title("Árbol Causal: Descubrimiento Automático de Desigualdad en el 401(k)", fontsize=16, fontweight='bold')
plt.tight_layout()

# Guardamos la imagen
plt.savefig('arbol_causal_401k.png', dpi=300, bbox_inches='tight')
print("📸 ¡Árbol Causal generado exitosamente y guardado como 'arbol_causal_401k.png'!")
plt.show()


# =============================================================================
# EMPAQUETADO AUTOMÁTICO EN EL ZIP (CON TU IMAGEN INCLUIDA)
# =============================================================================
print("\n📦 INICIANDO EMPAQUETADO EN ARCHIVO COMPRIMIDO...")

# Actualizar reportes de texto antes de comprimir
try:
    with open('tabla_1_iv_clasico.txt', 'w', encoding='utf-8') as f:
        f.write(str(modelo_iv.summary))
    with open('tabla_2_dml_rf.txt', 'w', encoding='utf-8') as f:
        f.write(dml_pliv_rf.summary.to_string())
    with open('tabla_3_dml_xgb.txt', 'w', encoding='utf-8') as f:
        f.write(dml_pliv_xgb.summary.to_string())
    print("✅ Tablas de regresión TXT preparadas.")
except NameError:
    print("ℹ️ Nota: Algunos modelos clásicos no están en la sesión, se usarán los TXT existentes.")

nombre_zip = 'paquete_final_proyecto_401k.zip'

# Lista con tus archivos y tu imagen exacta
todos_los_archivos = [
    'tabla_descriptiva_anexo.csv',
    'tabla_heterogenea.csv',
    'graficos_paper_401k.png',
    'arbol_causal_401k.png',  # <--- Tu imagen del árbol sin alteraciones
    'tabla_1_iv_clasico.txt',
    'tabla_2_dml_rf.txt',
    'tabla_3_dml_xgb.txt'
]

archivos_reales = [f for f in todos_los_archivos if os.path.exists(f)]

if len(archivos_reales) > 0:
    print(f"🗂️  Se encontraron {len(archivos_reales)} archivos para guardar.")
    with zipfile.ZipFile(nombre_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for archivo in archivos_reales:
            if archivo.endswith('.csv'):
                ruta_interna = os.path.join('datos_csv', archivo)
            elif archivo.endswith('.png'):
                ruta_interna = os.path.join('graficos_png', archivo)
            elif archivo.endswith('.txt'):
                ruta_interna = os.path.join('regresiones_txt', archivo)
            else:
                ruta_interna = archivo

            zipf.write(archivo, ruta_interna)
            print(f"   ➕ Añadido al ZIP: {archivo} -> {ruta_interna}")

    print(f"\n📥 Descargando paquete único: '{nombre_zip}'")
    files.download(nombre_zip)
else:
    print("❌ Error: No se encontraron archivos para comprimir.")